# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khaled-dragon/ML-intern/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row for my lane = one content item (content_hash_id), for one client
(client_hash_id), aggregated over a 30-day feature window ending mid-panel.
I'll use `fact_content_daily_performance` (aggregated to content level) plus
`fact_content_query_90d` for query-mix signals. Time window: I'll iterate on
`month=2026-03` (a mid-panel month), and treat the final month as a sealed
test month I don't touch yet.

In [1]:
%pip -q install duckdb huggingface_hub
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your HF READ token (hf_...): ··········


### "###############################################################################"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Features:** imp_prev30, avg_position (30d), visible_queries, rare_share, top_query_share
- **Label:** is_declining (imp_last30 < 0.8 × imp_prev30)
- **Context (not a feature, just for reading results):** client_hash_id, content_hash_id
- **Excluded:** the `_sample` table (it's the final month, June 2026 — using it now would leak
  the future into label design). Also excluding clients with less than 60 days of history in
  this window, since a 30/30 split needs both halves present.

### "###############################################################################"

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df()
print(schema[['column_name', 'column_type']].to_string())

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

In [3]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f"Duplicate (client, content, date) combos: {len(grain_check)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) combos: 0


In [4]:
size_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS start_d, MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(size_check)

   row_count    start_d      end_d
0    9841378 2026-03-01 2026-03-31


In [5]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows
0     9841378             3611061


In [6]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
    ),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_impressions ELSE 0 END) AS imp_prev30,
               AVG(CASE WHEN report_date > b.end_d - INTERVAL 30 DAY
                        THEN gsc_avg_position END) AS avg_position_prev30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.month = '2026-03'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f"{len(features):,} content items with enough history")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

100,849 content items with enough history


,client_hash_id,content_hash_id,imp_prev30,avg_position_prev30
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1120.0,4.429042
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,142.0,6.567020
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1366.0,6.421924
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2693.0,4.419866
4,client_73cda7b4e4f265ea,content_2662845f598544ef,141.0,6.303861


In [7]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count) AS visible_queries,
           ANY_VALUE(rare_impressions_share)       AS rare_share,
           MAX(impressions_90d)                    AS top_query_impressions,
           SUM(impressions_90d)                    AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f"joined: {len(data):,} rows")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 100,849 rows


,client_hash_id,content_hash_id,imp_prev30,avg_position_prev30,visible_queries,rare_share,top_query_impressions,kept_impressions,top_query_share
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1120.0,4.429042,8.0,0.028765,87.0,237.0,0.367089
1,client_73cda7b4e4f265ea,content_905aa32a0230694e,142.0,6.567020,NaN,NaN,NaN,NaN,NaN
2,client_73cda7b4e4f265ea,content_05434271b257bb68,1366.0,6.421924,18.0,0.175435,209.0,897.0,0.232999
3,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2693.0,4.419866,54.0,0.059806,1533.0,4909.0,0.312284
4,client_73cda7b4e4f265ea,content_2662845f598544ef,141.0,6.303861,NaN,NaN,NaN,NaN,NaN


In [8]:
outcome = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04'
    GROUP BY 1
""").df()

data = data.merge(outcome, on='content_hash_id', how='inner')
data['is_declining'] = (data['imp_next30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'avg_position_prev30', 'visible_queries', 'rare_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

from sklearn.linear_model import LogisticRegression

m_honest = LogisticRegression(max_iter=1000).fit(X, y)
print("Honest score (no leak):", m_honest.score(X, y))

model_data = model_data.copy()
model_data['imp_change_pct'] = (data['imp_next30'] - data['imp_prev30']) / data['imp_prev30']
X_leak = model_data[feature_cols + ['imp_change_pct']]

m_leak = LogisticRegression(max_iter=1000).fit(X_leak, y)
print("Score WITH leak:", m_leak.score(X_leak, y))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Honest score (no leak): 0.6753667431807324
Score WITH leak: 0.999032893280356


Adding `imp_change_pct` (computed directly from imp_next30, the same column
that defines is_declining) pushed the score from 0.675 to 0.999. That's the
leakage lesson from notebook 02, reproduced on real warehouse data: the
column doesn't teach the model a real pattern, it just hands it the answer.
I removed it and kept 0.675 as the honest baseline.

### "###############################################################################"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me about seasonality beyond one month, or about newer
clients who don't have both a prior-30 and next-30 window yet. Only 3.6M of
9.8M rows (about 37%) have gsc_data_available = TRUE, so a large share of
March's rows carry no real Search Console signal, meaning any feature built
without checking this flag first would silently include empty/non-informative
rows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.